In [ ]:
# Notebook focused on cleaning the dataset

In [ ]:
# Step 0: Uncompress the zipfile for NIH and Chexpert (You can skip this little part if you already have done this)

In [1]:
# Import the libraries needed to extract the information
import zipfile
import os
from tqdm import tqdm

In [2]:
# Function to extract the dataset archive
def extract_dataset(zip_path, extract_to):

    zip_path = str(os.path.expanduser(zip_path))
    extract_to = str(os.path.expanduser(extract_to))

    # First, we check if the file exists
    if not os.path.exists(zip_path):
        print(f"Error: File not found at {zip_path}")
        return
    # Create the destination directory if it doesn't exist
    print(f"Opening archive: {os.path.basename(zip_path)}")
    try:
        # Create the destination directory if it doesn't exist
        if not os.path.exists(extract_to):
            os.makedirs(extract_to, exist_ok=True)
            print(f"Directory created at: {extract_to}")

        # Extraction process with a progress bar
        with zipfile.ZipFile(zip_path, mode='r') as zip_ref:
            files = zip_ref.namelist()
            print(f"Extracting {len(files)} files...")
            for file in tqdm(files, desc="Extracting", unit="file"):
                zip_ref.extract(member=file, path=extract_to)
        print(f"\nSuccessfully extracted to: {extract_to}")

    except zipfile.BadZipFile:
        print(f"Error: The file at {zip_path} is corrupted.")

In [3]:
# Helper function used to manage Vena's datasets (NIH and CheXpert)
def setup_vena_datasets():
    # Extract CheXpert
    chexpert_zip = os.getenv("CHEXPERT_ZIP", "~/Downloads/Chexpert_Dataset.zip") # Change the name if you need it
    chexpert_out = os.getenv("CHEXPERT_OUT", "~/Vena/Datasets_V1/Chexpert_Raw")
    extract_dataset(chexpert_zip, chexpert_out)

    # Extract NIH
    nih_zip = os.getenv("NIH_ZIP", "~/Downloads/NIH_Dataset.zip") # Change the name if you need it
    nih_out = os.getenv("NIH_OUT", "~/Vena/Datasets_V1/NIH_Raw")
    extract_dataset(nih_zip, nih_out)

if __name__ == "__main__":
    setup_vena_datasets()

Opening archive: Chexpert_Dataset.zip
Directory created at: /home/suwy/Vena/Datasets_V1/Chexpert_Raw
Extracting 223651 files...


Extracting: 100%|██████████| 223651/223651 [02:04<00:00, 1791.55file/s]



Successfully extracted to: /home/suwy/Vena/Datasets_V1/Chexpert_Raw
Opening archive: NIH_Dataset.zip
Directory created at: /home/suwy/Vena/Datasets_V1/NIH_Raw
Extracting 112128 files...


Extracting: 100%|██████████| 112128/112128 [04:05<00:00, 456.95file/s]


Successfully extracted to: /home/suwy/Vena/Datasets_V1/NIH_Raw


In [15]:
# Organize the NIH images

In [16]:
import os
import shutil
import multiprocessing
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor # Cambiado a ThreadPool

In [17]:
# Even in notebooks, we can use the core count for threading
NUM_CORES = multiprocessing.cpu_count()

In [18]:
def move_single_file(args):
    """Helper function for parallel file movement"""
    src_path, dest_path = args
    try:
        shutil.move(src_path, dest_path)
        return True
    except Exception:
        return False

In [19]:
def flatten_nih_images(base_path, target_subdir="NIH_samples"):
    # Standardize paths
    base_path = str(os.path.expanduser(base_path))
    target_dir = os.path.join(base_path, target_subdir)

    if not os.path.exists(base_path):
        print(f"Error: Directory not found at {base_path}")
        return

    print(f"--- Starting Multi-Threaded Organization (Notebook Optimized) ---")
    print(f"Threads assigned: {NUM_CORES * 2}") # Threads can be higher than cores for I/O

    try:
        if not os.path.exists(target_dir):
            os.makedirs(target_dir, exist_ok=True)

        extensions = (".png", ".jpg", ".jpeg")
        tasks = []

        print("Scanning folder structure...")
        for root, dirs, files in os.walk(base_path):
            if target_subdir in root:
                continue
            for file in files:
                if file.lower().endswith(extensions):
                    src_path = os.path.join(root, file)
                    dest_path = os.path.join(target_dir, file)
                    tasks.append((src_path, dest_path))

        if not tasks:
            print("No images found to move.")
            return

        # Using ThreadPoolExecutor instead of ProcessPoolExecutor
        # For I/O tasks like moving 112k files, this is actually very efficient
        with ThreadPoolExecutor(max_workers=NUM_CORES * 2) as executor:
            list(tqdm(
                executor.map(move_single_file, tasks),
                total=len(tasks),
                desc="Flattening Dataset",
                unit="file"
            ))

        print(f"\nSuccessfully flattened to: {target_dir}")

    except Exception as e:
        print(f"Critical error during movement: {e}")

In [20]:
nih_path = "~/Vena/Datasets_V1/NIH_Raw"
flatten_nih_images(nih_path)

--- Starting Multi-Threaded Organization (Notebook Optimized) ---
Threads assigned: 8
Scanning folder structure...


Flattening Dataset: 100%|██████████| 112120/112120 [00:08<00:00, 13796.35file/s]


Successfully flattened to: /home/suwy/Vena/Datasets_V1/NIH_Raw/NIH_samples


In [22]:
from pathlib import Path
def cleanup_nih_folders(base_path):
    # Estandarizamos la ruta
    base_path = Path(base_path).expanduser()

    print(f"Cleaning up directories in: {base_path}")

    # Buscamos todas las carpetas que empiecen con 'images_'
    # Pero ignoramos 'NIH_samples' por seguridad
    folders_to_delete = [f for f in base_path.glob("images_*") if f.is_dir()]

    if not folders_to_delete:
        print("No folders found to delete.")
        return

    for folder in folders_to_delete:
        try:
            # Borra la carpeta y su contenido (como rm -rf)
            shutil.rmtree(folder)
            print(f"Deleted: {folder.name}")
        except Exception as e:
            print(f"Error deleting {folder.name}: {e}")

    print("\nCleanup finished. The dataset is now clean.")

In [23]:
# Ejecución
nih_path = "~/Vena/Datasets_V1/NIH_Raw"
cleanup_nih_folders(nih_path)

Cleaning up directories in: /home/suwy/Vena/Datasets_V1/NIH_Raw
Deleted: images_001
Deleted: images_002
Deleted: images_003
Deleted: images_004
Deleted: images_005
Deleted: images_006
Deleted: images_007
Deleted: images_008
Deleted: images_009
Deleted: images_010
Deleted: images_011
Deleted: images_012

Cleanup finished. The dataset is now clean.


In [27]:
!ls -F ~/Vena/Datasets_V1/NIH_Raw

ARXIV_V5_CHESTXRAY.pdf	FAQ_CHESTXRAY.pdf  README_CHESTXRAY.pdf
BBox_List_2017.csv	LOG_CHESTXRAY.pdf  test_list.txt
Data_Entry_2017.csv	NIH_samples/	   train_val_list.txt


In [28]:
!rm ~/Vena/Datasets_V1/NIH_Raw/*_CHESTXRAY.pdf